# Lab 3: Multi-Source Retail Sales Data Integration and Analysis

## Problem Statement

A multinational retail company maintains its business data across multiple systems.
Sales transactions are stored in CSV format, product information in JSON format,
and customer information in Excel format.

The objective of this analysis is to import, clean, integrate, and analyze these
heterogeneous datasets using R. The analysis focuses on sales revenue, top-performing
products and countries, customer purchase value, and customer value classification.

The final integrated dataset will also be used for SQL-based analysis using SQLite.

In [5]:
install.packages("RSQLite")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [6]:
library(RSQLite)

In [7]:
install.packages(c("DBI", "RSQLite"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [8]:
library(DBI)
library(RSQLite)

In [9]:
# Install packages if required
# install.packages(c("tidyverse", "jsonlite", "readxl", "DBI", "RSQLite"))

# Load required libraries
library(tidyverse)
library(jsonlite)
library(readxl)
library(DBI)
library(RSQLite)

In [10]:
# Import transactions CSV
transactions <- read_csv("/content/transactions.csv")

# Display first few records
head(transactions)

Rows: 541909 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): InvoiceNo, StockCode
dbl  (2): CustomerID, Quantity
dttm (1): InvoiceDate

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


In [11]:
str(transactions)

spc_tbl_ [541,909 × 5] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 - attr(*, "spec")=
  .. cols(
  ..   InvoiceNo = col_character(),
  ..   StockCode = col_character(),
  ..   CustomerID = col_double(),
  ..   Quantity = col_double(),
  ..   InvoiceDate = col_datetime(format = "")
  .. )
 - attr(*, "problems")=<pointer: 0x5c79bf80f1f0> 


In [12]:
dim(transactions)

[1] 541909      5

In [13]:
# Import products JSON
products <- fromJSON("/content/products.json")

# Convert to data frame
products <- as.data.frame(products)

# Display first few records
head(products)

,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,22752,SET 7 BABUSHKA NESTING BOXES,7.65


In [14]:
str(products)

'data.frame':	4070 obs. of  3 variables:
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...


In [15]:
dim(products)

[1] 4070    3

In [16]:
# Import customers Excel file
customers <- read_excel("customers.xlsx")

# Display first few records
head(customers)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [17]:
str(customers)

tibble [4,373 × 2] (S3: tbl_df/tbl/data.frame)
 $ CustomerID: num [1:4373] 17850 13047 12583 13748 15100 ...
 $ Country   : chr [1:4373] "United Kingdom" "United Kingdom" "France" "United Kingdom" ...


In [18]:
dim(customers)

[1] 4373    2

In [19]:
# Structure of transactions
str(transactions)

# Structure of products
str(products)

# Structure of customers
str(customers)

spc_tbl_ [541,909 × 5] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 - attr(*, "spec")=
  .. cols(
  ..   InvoiceNo = col_character(),
  ..   StockCode = col_character(),
  ..   CustomerID = col_double(),
  ..   Quantity = col_double(),
  ..   InvoiceDate = col_datetime(format = "")
  .. )
 - attr(*, "problems")=<pointer: 0x5c79bf80f1f0> 
'data.frame':	4070 obs. of  3 variables:
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25

In [20]:
# Column names
colnames(transactions)
colnames(products)
colnames(customers)

[1] "InvoiceNo"   "StockCode"   "CustomerID"  "Quantity"    "InvoiceDate"

[1] "StockCode"   "Description" "UnitPrice"

[1] "CustomerID" "Country"

In [21]:
summary(transactions)
summary(products)
summary(customers)

     InvoiceNo          StockCode        CustomerID        Quantity         
 Length   :541909   Length   :541909   Min.   :12346    Min.   :-80995.000  
 N.unique : 25900   N.unique :  4070   1st Qu.:13953    1st Qu.:     1.000  
 N.blank  :     0   N.blank  :     0   Median :15152    Median :     3.000  
 Min.nchar:     6   Min.nchar:     1   Mean   :15288    Mean   :     9.552  
 Max.nchar:     7   Max.nchar:    12   3rd Qu.:16791    3rd Qu.:    10.000  
                                       Max.   :18287    Max.   : 80995.000  
                                       NAs    :135080                       
  InvoiceDate                 
 Min.   :2010-12-01 08:26:00  
 1st Qu.:2011-03-28 11:34:00  
 Median :2011-07-19 17:17:00  
 Mean   :2011-07-04 13:34:57  
 3rd Qu.:2011-10-19 11:27:00  
 Max.   :2011-12-09 12:50:00  
                              

     StockCode       Description     UnitPrice        
 Length   :4070   Length   :4070   Min.   :    0.000  
 N.unique :4070   N.unique :3753   1st Qu.:    1.250  
 N.blank  :   0   N.blank  :   0   Median :    2.510  
 Min.nchar:   1   Min.nchar:   1   Mean   :    6.905  
 Max.nchar:  12   Max.nchar:  35   3rd Qu.:    4.250  
                  NAs      : 176   Max.   :11062.060  

   CustomerID         Country    
 Min.   :12346   Length   :4373  
 1st Qu.:13813   N.unique :  37  
 Median :15300   N.blank  :   0  
 Mean   :15300   Min.nchar:   3  
 3rd Qu.:16778   Max.nchar:  20  
 Max.   :18287                   
 NAs    :1                       

In [22]:
head(transactions)
head(products)
head(customers)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,22752,SET 7 BABUSHKA NESTING BOXES,7.65


CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [23]:
# Missing values in transactions
colSums(is.na(transactions))

# Missing values in products
colSums(is.na(products))

# Missing values in customers
colSums(is.na(customers))

InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0

StockCode Description   UnitPrice 
          0         176           0

CustomerID    Country 
         1          0

In [24]:
# Number of duplicate rows
sum(duplicated(transactions))

sum(duplicated(products))

sum(duplicated(customers))

[1] 5429

[1] 0

[1] 0

In [26]:
transactions <- transactions %>% distinct()

products <- products %>% distinct()

customers <- customers %>% distinct()

In [27]:
# Check invalid/zero quantities
transactions %>%
  filter(Quantity <= 0)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
C536379,D,14527,-1,2010-12-01 09:41:00
C536383,35004C,15311,-1,2010-12-01 09:49:00
C536391,22556,17548,-12,2010-12-01 10:24:00
C536391,21984,17548,-24,2010-12-01 10:24:00
C536391,21983,17548,-24,2010-12-01 10:24:00
C536391,21980,17548,-24,2010-12-01 10:24:00
C536391,21484,17548,-12,2010-12-01 10:24:00
C536391,22557,17548,-12,2010-12-01 10:24:00
C536391,22553,17548,-24,2010-12-01 10:24:00


In [28]:
transactions <- transactions %>%
  filter(Quantity > 0)

In [29]:
products %>%
  filter(UnitPrice <= 0)

StockCode,Description,UnitPrice
<chr>,<chr>,<dbl>
21134,NA,0
22145,NA,0
37509,NA,0
85226A,NA,0
85044,NA,0
20950,NA,0
37461,NA,0
84670,NA,0
84952C,NA,0


In [30]:
products <- products %>%
  filter(UnitPrice > 0)

In [31]:
sum(is.na(transactions$CustomerID))

[1] 133259

In [32]:
transactions <- transactions %>%
  filter(!is.na(CustomerID))

In [33]:
sum(is.na(products$StockCode))
sum(is.na(products$Description))
sum(is.na(products$UnitPrice))

[1] 0

[1] 0

[1] 0

In [34]:
products <- products %>%
  filter(
    !is.na(StockCode),
    !is.na(Description),
    !is.na(UnitPrice)
  )

In [35]:
sum(is.na(customers$CustomerID))
sum(is.na(customers$Country))

[1] 1

[1] 0

In [36]:
customers <- customers %>%
  filter(
    !is.na(CustomerID),
    !is.na(Country)
  )

In [37]:
transactions_with_products <- transactions %>%
  left_join(
    products %>% select(StockCode, UnitPrice),
    by = "StockCode"
  )

In [38]:
transactions_with_products <- transactions_with_products %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

In [39]:
head(transactions_with_products)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,UnitPrice,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<dbl>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,2.55,15.30
536365,71053,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,2.75,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,22752,17850,2,2010-12-01 08:26:00,7.65,15.30


In [40]:
retail_sales <- transactions %>%
  left_join(
    products,
    by = "StockCode"
  ) %>%
  left_join(
    customers,
    by = "CustomerID"
  ) %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

In [41]:
head(retail_sales)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<chr>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom,15.30


In [42]:
str(retail_sales)

tibble [392,708 × 9] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:392708] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:392708] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:392708] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:392708] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:392708], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ Description: chr [1:392708] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ UnitPrice  : num [1:392708] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ Country    : chr [1:392708] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...
 $ Revenue    : num [1:392708] 15.3 20.3 22 20.3 20.3 ...


### Join Justification

A left join was selected because the transaction dataset represents the primary
sales records. Using a left join ensures that all transaction records are retained
while matching product and customer information is added wherever available.

In [43]:
# Number of rows and columns
dim(retail_sales)

[1] 392708      9

In [44]:
cat("Number of rows:", nrow(retail_sales), "\n")
cat("Number of columns:", ncol(retail_sales), "\n")

Number of rows: 392708 
Number of columns: 9 


In [45]:
glimpse(retail_sales)

Rows: 392,708
Columns: 9
$ InvoiceNo   <chr> "536365", "536365", "536365", "536365", "536365", "536365"…
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ CustomerID  <dbl> 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17…
$ Quantity    <dbl> 6, 6, 8, 6, 6, 2, 6, 6, 6, 32, 6, 6, 8, 6, 6, 3, 2, 3, 3, …
$ InvoiceDate <dttm> 2010-12-01 08:26:00, 2010-12-01 08:26:00, 2010-12-01 08:2…
$ Description <chr> "WHITE HANGING HEART T-LIGHT HOLDER", "WHITE METAL LANTERN…
$ UnitPrice   <dbl> 2.55, 3.39, 2.75, 3.39, 3.39, 7.65, 4.25, 1.85, 1.85, 1.69…
$ Country     <chr> "United Kingdom", "United Kingdom", "United Kingdom", "Uni…
$ Revenue     <dbl> 15.30, 20.34, 22.00, 20.34, 20.34, 15.30, 25.50, 11.10, 11…


In [46]:
unmatched_products <- transactions %>%
  anti_join(products, by = "StockCode")

nrow(unmatched_products)

[1] 4831

In [47]:
head(unmatched_products)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536783,84952C,15061,36,2010-12-02 15:19:00
536796,84952C,15574,1,2010-12-02 15:46:00
537042,21011,13838,2,2010-12-05 10:45:00
537133,21836,18156,1,2010-12-05 12:29:00
537136,84952C,12748,1,2010-12-05 12:42:00
537153,21696,16718,6,2010-12-05 13:03:00


In [48]:
unmatched_customers <- transactions %>%
  anti_join(customers, by = "CustomerID")

nrow(unmatched_customers)

[1] 0

In [49]:
head(unmatched_customers)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>


In [50]:
cat("Unmatched product records:",
    nrow(unmatched_products), "\n")

cat("Unmatched customer records:",
    nrow(unmatched_customers), "\n")

Unmatched product records: 4831 
Unmatched customer records: 0 


In [51]:
total_revenue <- retail_sales %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE)
  )

total_revenue

Total_Revenue
<dbl>
10752840


In [52]:
cat(
  "Total Sales Revenue =",
  round(total_revenue$Total_Revenue, 2)
)

Total Sales Revenue = 10752840

In [53]:
top_5_products <- retail_sales %>%
  group_by(StockCode, Description) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

top_5_products

StockCode,Description,Total_Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
47566,PARTY BUNTING,142437.56
22423,REGENCY CAKESTAND 3 TIER,135604.80
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


In [54]:
top_5_countries <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

top_5_countries

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


In [55]:
top_5_countries <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

top_5_countries

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


In [57]:
customer_revenue <- retail_sales %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  )

In [58]:
top_5_customers <- customer_revenue %>%
  arrange(desc(Total_Purchase)) %>%
  slice_head(n = 5)

top_5_customers

CustomerID,Total_Purchase
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [59]:
summary(customer_revenue$Total_Purchase)

     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
     1.25    361.00    802.60   2478.18   2000.64 408759.96 

In [60]:
quantile(
  customer_revenue$Total_Purchase,
  probs = c(0.25, 0.50, 0.75),
  na.rm = TRUE
)

25%      50%      75% 
 361.005  802.600 2000.640

In [61]:
q1 <- quantile(
  customer_revenue$Total_Purchase,
  0.25,
  na.rm = TRUE
)

q2 <- quantile(
  customer_revenue$Total_Purchase,
  0.50,
  na.rm = TRUE
)

q3 <- quantile(
  customer_revenue$Total_Purchase,
  0.75,
  na.rm = TRUE
)

In [62]:
customer_revenue <- customer_revenue %>%
  mutate(
    Customer_Value = case_when(
      Total_Purchase <= q1 ~ "Low Value",
      Total_Purchase <= q2 ~ "Medium Value",
      Total_Purchase <= q3 ~ "High Value",
      TRUE ~ "Premium"
    )
  )

In [63]:
head(customer_revenue)

CustomerID,Total_Purchase,Customer_Value
<dbl>,<dbl>,<chr>
12346,77183.60,Premium
12347,5438.44,Premium
12348,1790.16,High Value
12349,1926.96,High Value
12350,408.04,Medium Value
12352,1616.57,High Value


In [64]:
customer_revenue %>%
  count(Customer_Value)

Customer_Value,n
<chr>,<int>
High Value,1084
Low Value,1085
Medium Value,1085
Premium,1085


### Customer Value Classification

Customer value thresholds were determined using the 25th, 50th, and 75th
percentiles of total customer purchase value. This provides a data-driven
classification into Low Value, Medium Value, High Value, and Premium customers.

In [65]:
country_revenue <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue))

In [66]:
country_revenue

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857.13
Netherlands,363884.48
EIRE,331660.17
Germany,263818.97
France,226975.60
Australia,173918.61
Spain,67426.09
Switzerland,66619.97
Japan,48600.22


In [67]:
high_performing_market <- country_revenue %>%
  slice_max(
    Total_Revenue,
    n = 1
  )

high_performing_market

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857


In [68]:
underperforming_market <- country_revenue %>%
  slice_min(
    Total_Revenue,
    n = 1
  )

underperforming_market

Country,Total_Revenue
<chr>,<dbl>
Saudi Arabia,181.44


In [69]:
cat(
  "High-performing market:",
  high_performing_market$Country,
  "with revenue =",
  round(high_performing_market$Total_Revenue, 2),
  "\n"
)

cat(
  "Underperforming market:",
  underperforming_market$Country,
  "with revenue =",
  round(underperforming_market$Total_Revenue, 2),
  "\n"
)

High-performing market: United Kingdom with revenue = 8861857 
Underperforming market: Saudi Arabia with revenue = 181.44 


### Market Interpretation

The high-performing market is identified based on the highest total revenue
generated among the countries in the cleaned dataset. The underperforming
market is identified based on the lowest total revenue. These results can
help management identify markets that require continued investment and
markets that may require targeted strategies to improve sales performance.

In [70]:
str(transactions)
str(products)
str(customers)

tibble [392,708 × 5] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:392708] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:392708] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:392708] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:392708] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:392708], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
'data.frame':	3855 obs. of  3 variables:
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
tibble [4,372 × 2] (S3: tbl_df/tbl/data.frame)
 $ CustomerID: num [1:4372] 17850 13047 12583 13748 15100 ...
 $ Country   : chr [1:4372] "United Kingdom" "United Kingdom" "France" "United Kingdom" ...


In [73]:
install.packages("RSQLite")
install.packages("DBI")

library(DBI)
library(RSQLite)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [74]:
con <- dbConnect(SQLite(), "retail_sales.db")

In [77]:
dbListTables(con)

character(0)

In [78]:
# Export the retail_sales dataframe to SQLite
dbWriteTable(
  con,
  "retail_sales",
  retail_sales,
  overwrite = TRUE
)

In [79]:
dbListTables(con)

[1] "retail_sales"

In [80]:
dbGetQuery(
  con,
  "PRAGMA table_info(retail_sales)"
)

cid,name,type,notnull,dflt_value,pk
<int>,<chr>,<chr>,<int>,<lgl>,<int>
0,InvoiceNo,TEXT,0,NA,0
1,StockCode,TEXT,0,NA,0
2,CustomerID,REAL,0,NA,0
3,Quantity,REAL,0,NA,0
4,InvoiceDate,REAL,0,NA,0
5,Description,TEXT,0,NA,0
6,UnitPrice,REAL,0,NA,0
7,Country,TEXT,0,NA,0
8,Revenue,REAL,0,NA,0


In [81]:
dbGetQuery(
  con,
  "SELECT * FROM retail_sales LIMIT 5"
)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>
536365,85123A,17850,6,1291191960,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
536365,71053,17850,6,1291191960,WHITE METAL LANTERN,3.39,United Kingdom,20.34
536365,84406B,17850,8,1291191960,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
536365,84029G,17850,6,1291191960,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
536365,84029E,17850,6,1291191960,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34


In [82]:
query1 <- "
SELECT
    CustomerID,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5;
"

top_5_customers_sql <- dbGetQuery(con, query1)

top_5_customers_sql

CustomerID,Total_Revenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [83]:
cat("Top 5 Customers Based on Revenue\n")

top_5_customers_sql

Top 5 Customers Based on Revenue


CustomerID,Total_Revenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [84]:
query2 <- "
SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC;
"

country_revenue_sql <- dbGetQuery(con, query2)

country_revenue_sql

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857.13
Netherlands,363884.48
EIRE,331660.17
Germany,263818.97
France,226975.60
Australia,173918.61
Spain,67426.09
Switzerland,66619.97
Japan,48600.22


In [85]:
query2 <- "
SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC
LIMIT 5;
"

top_5_countries_sql <- dbGetQuery(con, query2)

top_5_countries_sql

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


In [86]:
# Highest revenue country
highest_country <- country_revenue %>%
  slice_max(Total_Revenue, n = 1)

# Lowest revenue country
lowest_country <- country_revenue %>%
  slice_min(Total_Revenue, n = 1)

# Highest revenue product
highest_product <- top_5_products %>%
  slice_max(Total_Revenue, n = 1)

# Highest value customer
highest_customer <- top_5_customers %>%
  slice_max(Total_Purchase, n = 1)

# Display results
highest_country
lowest_country
highest_product
highest_customer

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857


Country,Total_Revenue
<chr>,<dbl>
Saudi Arabia,181.44


StockCode,Description,Total_Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.6


CustomerID,Total_Purchase
<dbl>,<dbl>
18102,408760


In [87]:
cat(
  "Insight 1: The highest-value customer generated revenue of",
  round(highest_customer$Total_Purchase, 2),
  ". This indicates that high-value customers can make a significant contribution to overall sales and should be targeted with customer retention strategies.\n"
)

Insight 1: The highest-value customer generated revenue of 408760 . This indicates that high-value customers can make a significant contribution to overall sales and should be targeted with customer retention strategies.


In [88]:
cat(
  "Insight 2: The highest-revenue product was",
  highest_product$Description,
  "with revenue of",
  round(highest_product$Total_Revenue, 2),
  ". This product is an important contributor to sales and may require appropriate inventory and marketing attention.\n"
)

Insight 2: The highest-revenue product was PAPER CRAFT , LITTLE BIRDIE with revenue of 168469.6 . This product is an important contributor to sales and may require appropriate inventory and marketing attention.


In [89]:
cat(
  "Insight 3: The highest-revenue country was",
  highest_country$Country,
  "with revenue of",
  round(highest_country$Total_Revenue, 2),
  ". This indicates that the market is performing strongly and may represent an important market for continued business investment.\n"
)

Insight 3: The highest-revenue country was United Kingdom with revenue of 8861857 . This indicates that the market is performing strongly and may represent an important market for continued business investment.


## 3 Business Insights

1.  **Customer Insight:** The highest-value customer generated revenue of 408760. This indicates that high-value customers can make a significant contribution to overall sales and should be targeted with customer retention strategies.

2.  **Product Insight:** The highest-revenue product was PAPER CRAFT , LITTLE BIRDIE with revenue of 168469.6. This product is an important contributor to sales and may require appropriate inventory and marketing attention.

3.  **Market Insight:** The highest-revenue country was United Kingdom with revenue of 8861857. This indicates that the market is performing strongly and may represent an important market for continued business investment.

In [90]:
# Close SQLite connection
dbDisconnect(con)

In [91]:
dbIsValid(con)

[1] FALSE

# Conclusion

The analysis successfully integrated retail transaction, product, and customer
information from CSV, JSON, and Excel sources using R. The datasets were
cleaned by handling missing values, duplicate records, invalid quantities,
and invalid prices. A Revenue attribute was created using Quantity × UnitPrice.

The integrated dataset was analyzed to identify total sales revenue, the
top-performing products and countries, and the highest-value customers.
Customers were also classified into Low Value, Medium Value, High Value,
and Premium categories using data-driven purchase-value thresholds.

The final integrated dataset was stored in a SQLite database in a table named
`retail_sales`, and SQL queries were executed to retrieve meaningful business
information. The analysis provides useful insights into customer value,
product performance, and market performance that can support data-driven
retail business decisions.